# Self-Consistency Evaluation — llama-3.3-70b + PROMPT_FULL

Compares extraction accuracy for a single model and prompt across two decoding strategies.

| | Baseline (temp=0, k=1) | Self-Consistency (temp=0.7, k=5) |
|---|---|---|
| **llama-3.3-70b + PROMPT_FULL** | ? | ? |

**Hypothesis**: SC aggregation over k=5 samples improves field-level accuracy over a single deterministic call.

In [12]:
# !pip install langchain-ollama
# import sys
# !{sys.executable} -m pip install langchain-openai
#!pip install langchain-google-genai

## Cell 1 — Schema definitions
Copied from `interpretation_agent.ipynb`. Keep in sync if the main notebook changes.

In [13]:
import os
import re
import json
import time
from typing import Optional, Union
from pydantic import BaseModel, Field, field_validator
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
load_dotenv()

class ExtractionResult(BaseModel):
    """Structured clinical facts extracted from a single caregiver message."""
    fever:               Optional[str]   = Field(None, description="'yes' or 'no'")
    temperature_f:       Optional[float] = Field(None, description="Temperature in °F if stated")
    fever_duration_days: Optional[float] = Field(None, description="Days fever has been present")
    vomiting:            Optional[str]   = Field(None, description="'none', 'once', or 'repeated'")
    alert:               Optional[str]   = Field(None, description="'yes' if child responds normally, 'no' if lethargic/hard to wake")
    drinking:            Optional[str]   = Field(None, description="'yes', 'some', or 'no'")
    urination_8h:        Optional[str]   = Field(None, description="'yes' or 'no' — has child urinated in last 8 hours")
    breathing_issues:    Optional[str]   = Field(None, description="'yes' or 'no'")
    rash:                Optional[str]   = Field(None, description="'yes' or 'no'")
    neck_pain:           Optional[str]   = Field(None, description="'yes' or 'no'")
    age_months:          Optional[int]   = Field(None, description="Child age in months")
    current_medication:  Optional[str]   = Field(None, description="Name of medication if mentioned")
    medication_last_dose:Optional[str]   = Field(None, description="When last dose was given")

    # yes/no string fields: coerce bool/int → "yes"/"no"
    @field_validator("fever", "alert", "urination_8h", "breathing_issues", "rash", "neck_pain", mode="before")
    @classmethod
    def coerce_bool_to_yesno(cls, v):
        if isinstance(v, (bool, int)):
            return "yes" if v else "no"
        return v

    @field_validator("drinking", mode="before")
    @classmethod
    def coerce_bool_drinking(cls, v):
        if isinstance(v, (bool, int)):
            return "yes" if v else "no"
        return v

    @field_validator("vomiting", mode="before")
    @classmethod
    def coerce_bool_vomiting(cls, v):
        if isinstance(v, (bool, int)):
            return "once" if v else "none"
        return v

    # Free-text fields: bool/int doesn't map to a meaningful string — treat as null
    @field_validator("current_medication", "medication_last_dose", mode="before")
    @classmethod
    def coerce_bool_freetext(cls, v):
        if isinstance(v, (bool, int)):
            return None
        return v

    # numeric fields: coerce string → correct type
    @field_validator("temperature_f", "fever_duration_days", mode="before")
    @classmethod
    def coerce_float(cls, v):
        if isinstance(v, str):
            try: return float(v)
            except (ValueError, TypeError): return None
        return v

    @field_validator("age_months", mode="before")
    @classmethod
    def coerce_int(cls, v):
        if isinstance(v, str):
            try: return int(float(v))
            except (ValueError, TypeError): return None
        return v

# Clinical fields to compare (excludes agent-logic fields)
CLINICAL_FIELDS = [
    "fever", "temperature_f", "fever_duration_days", "vomiting",
    "alert", "drinking", "urination_8h", "breathing_issues",
    "rash", "neck_pain", "age_months", "current_medication", "medication_last_dose"
]

REQUIRED_FIELDS_ORDER = ["alert", "breathing_issues", "temperature_f", "urination_8h", "drinking", "current_medication"]
QUESTION_MAP = {
    "alert":              "Is your child awake and responding normally when you talk to them?",
    "breathing_issues":   "Is your child having any trouble breathing or breathing fast?",
    "temperature_f":      "What is the temperature right now?",
    "urination_8h":       "Has your child urinated in the last 8 hours?",
    "drinking":           "Is your child drinking any fluids?",
    "current_medication": "Is your child currently taking any medications?",
}

print("Schema definitions loaded.")

Schema definitions loaded.


## Cell 2 — Model configs
`llm_baseline` (temp=0) for the single deterministic call.  
`llm_sc` (temp=0.7) for the k=5 stochastic samples used by SC.

In [14]:
from langchain_groq import ChatGroq

GROQ_API_KEY = os.environ["GROQ_API_KEY"]

# Deterministic baseline
llm_baseline = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=GROQ_API_KEY, temperature=0)

# Stochastic sampler for SC
llm_sc = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=GROQ_API_KEY, temperature=0.7)

print("Models ready: llama-3.3-70b-versatile  (temp=0 baseline, temp=0.7 SC sampler)")

Models ready: llama-3.3-70b-versatile  (temp=0 baseline, temp=0.7 SC sampler)


## Cell 3 — Prompt

Using **PROMPT_FULL** for both methods — keeps the prompt constant so only the decoding strategy differs.

In [15]:
PROMPT_FULL = """You are a clinical information extractor for a pediatric triage system.

YOUR ONLY JOB is to extract structured clinical facts from the caregiver's message.

STRICT RULES:
1. Extract ONLY what the caregiver explicitly states or clearly implies.
2. Set a field to null if the message provides NO new information about it.
3. Do NOT infer, guess, or fill in fields that were not mentioned.
4. Do NOT generate clinical advice, diagnosis, treatment, or disposition — ever.
5. Age conversion: if given in years, convert to months (e.g. "6 years" → 72).
6. Temperature conversion: if given in Celsius, convert to Fahrenheit.
7. Vomiting: 'none' = no vomiting, 'once' = one episode, 'repeated' = more than once.
"""

print("PROMPT_FULL defined.")

PROMPT_FULL defined.


## Cell 4 — Load dataset + evaluation helpers

In [16]:
with open("golden_dataset.json") as f:
    dataset = json.load(f)

targeted_cases = dataset["targeted_cases"]
scenarios      = dataset["scenarios"]
print(f"Loaded {len(targeted_cases)} targeted cases, {len(scenarios)} scenarios.")


def run_extraction(caregiver_message: str, history: list, llm, system_prompt: str) -> ExtractionResult:
    """
    Call the LLM directly (no tool calling) to avoid Groq's strict schema validation.
    Parses JSON from the response text and validates with Pydantic.
    Works across all models including DeepSeek-R1 (strips <think> tokens).
    """
    field_list = ", ".join(f'"{f}"' for f in CLINICAL_FIELDS)
    json_instruction = (
        f"\n\nRespond with ONLY a valid JSON object containing these exact fields: {field_list}. "
        "Set any field to null if the message does not mention it."
    )

    prompt_messages = [SystemMessage(content=system_prompt + json_instruction)]
    if history:
        history_text = "\n".join(
            f"{'Caregiver' if role == 'caregiver' else 'Agent'}: {content}"
            for role, content in history
        )
        prompt_messages.append(SystemMessage(content=f"Conversation so far:\n{history_text}"))
    prompt_messages.append(HumanMessage(content=f"Latest message: {caregiver_message}"))

    response = llm.invoke(prompt_messages)
    text = response.content

    # Strip DeepSeek-R1 <think>...</think> reasoning tokens
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Strip markdown code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text.strip())
    # Extract the first JSON object found
    match = re.search(r"\{.*\}", text, re.DOTALL)
    data = json.loads(match.group()) if match else {}

    return ExtractionResult.model_validate(data)


def simulate_merge(prior_state: dict, extracted: ExtractionResult) -> dict:
    """Merge extracted fields into prior_state, only updating None fields."""
    merged = dict(prior_state)
    for field in CLINICAL_FIELDS:
        new_val = getattr(extracted, field, None)
        if new_val is not None and merged.get(field) is None:
            merged[field] = new_val
    return merged


def compare_fields(expected: dict, actual_source) -> dict:
    """Compare expected vs actual per field. Returns {field: {expected, actual, match}}."""
    results = {}
    for field in CLINICAL_FIELDS:
        if field not in expected:
            continue
        exp = expected[field]
        act = actual_source.get(field) if isinstance(actual_source, dict) else getattr(actual_source, field, None)
        results[field] = {"expected": exp, "actual": act, "match": exp == act}
    return results


def score(comparison: dict) -> tuple:
    correct = sum(1 for r in comparison.values() if r["match"])
    return correct, len(comparison)


def next_required_field(state: dict) -> Optional[str]:
    return next((f for f in REQUIRED_FIELDS_ORDER if state.get(f) is None), None)


print("Helpers defined.")

Loaded 10 targeted cases, 2 scenarios.
Helpers defined.


## Cell 5 — Evaluate functions

`evaluate()` uses a single `run_extraction()` call per message (baseline).  
`evaluate_sc()` uses `run_extraction_sc()` — k calls aggregated per field (SC).  
Results stored in `result_baseline` / `result_sc` with wall-clock timings.

In [17]:
def evaluate(llm, system_prompt: str) -> dict:
    """Run full evaluation for one (model, prompt) combo."""
    targeted_results = []
    for case in targeted_cases:
        extracted = run_extraction(case["caregiver_message"], [], llm, system_prompt)
        ext_cmp   = compare_fields(case["expected_extracted_fields"], extracted)
        merged    = simulate_merge(case["prior_state"], extracted)
        cum_cmp   = compare_fields(case["expected_cumulative_state"], merged)
        targeted_results.append({
            "id":         case["id"],
            "category":   case["category"],
            "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1], "detail": ext_cmp},
            "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1], "detail": cum_cmp},
        })

    scenario_results = []
    for scenario in scenarios:
        state, history, turn_results = {f: None for f in CLINICAL_FIELDS}, [], []
        for turn in scenario["turns"]:
            msg       = turn["caregiver_message"]
            extracted = run_extraction(msg, history, llm, system_prompt)
            ext_cmp   = compare_fields(turn["expected_extracted_fields"], extracted)
            state     = simulate_merge(state, extracted)
            cum_cmp   = compare_fields(turn["expected_cumulative_state"], state)
            nf        = next_required_field(state)
            history.append(("caregiver", msg))
            if nf:
                history.append(("agent", QUESTION_MAP[nf]))
            turn_results.append({
                "turn":       turn["turn"],
                "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1]},
                "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1]},
            })
        scenario_results.append({"id": scenario["id"], "turns": turn_results})

    return {"targeted": targeted_results, "scenarios": scenario_results}

print("evaluate() defined.")

evaluate() defined.


In [18]:
# ── Run: Baseline (llama-3.3-70b, temp=0, PROMPT_FULL) ──────────────────────
print("Running baseline ...", end=" ", flush=True)
t0 = time.time()
result_baseline = evaluate(llm_baseline, PROMPT_FULL)
time_baseline   = time.time() - t0
c = sum(r["extraction"]["correct"] for r in result_baseline["targeted"])
t = sum(r["extraction"]["total"]   for r in result_baseline["targeted"])
print(f"{c}/{t}  ({time_baseline:.1f}s)")

Running baseline ... 120/130  (8.6s)


## Cell 5.SC — Self-Consistency: aggregation + evaluate_sc

`aggregate_sc()` merges k `ExtractionResult` samples field-by-field:
- **Categorical** (fever, alert, vomiting, drinking, urination_8h, breathing_issues, rash, neck_pain): majority vote over non-null values
- **Numeric** (temperature_f, fever_duration_days, age_months): median of non-null values
- **Free-text** (current_medication, medication_last_dose): most frequent non-null string (case-insensitive)
- If > half of k samples are null for a field → null

In [19]:
import statistics
from collections import Counter

SC_K = 5

CATEGORICAL_FIELDS = ["fever", "alert", "vomiting", "drinking", "urination_8h",
                       "breathing_issues", "rash", "neck_pain"]
NUMERIC_FIELDS     = ["temperature_f", "fever_duration_days", "age_months"]
FREETEXT_FIELDS    = ["current_medication", "medication_last_dose"]


def aggregate_sc(samples: list) -> ExtractionResult:
    """Aggregate k ExtractionResult samples field-by-field."""
    aggregated = {}
    for field in CLINICAL_FIELDS:
        values   = [getattr(s, field) for s in samples]
        non_null = [v for v in values if v is not None]
        if len(non_null) <= len(samples) / 2:
            aggregated[field] = None
            continue
        if field in CATEGORICAL_FIELDS:
            aggregated[field] = Counter(non_null).most_common(1)[0][0]
        elif field in NUMERIC_FIELDS:
            aggregated[field] = statistics.median(non_null)
        else:  # FREETEXT_FIELDS
            winner_lower = Counter(v.lower() for v in non_null).most_common(1)[0][0]
            aggregated[field] = next(v for v in non_null if v.lower() == winner_lower)
    return ExtractionResult.model_validate(aggregated)


def run_extraction_sc(caregiver_message: str, history: list, system_prompt: str) -> ExtractionResult:
    """k independent calls with llm_sc, then field-level aggregation."""
    samples = [run_extraction(caregiver_message, history, llm_sc, system_prompt)
               for _ in range(SC_K)]
    return aggregate_sc(samples)


def evaluate_sc(system_prompt: str) -> dict:
    """Run full evaluation using SC extraction (mirrors evaluate())."""
    targeted_results = []
    for case in targeted_cases:
        extracted = run_extraction_sc(case["caregiver_message"], [], system_prompt)
        ext_cmp   = compare_fields(case["expected_extracted_fields"], extracted)
        merged    = simulate_merge(case["prior_state"], extracted)
        cum_cmp   = compare_fields(case["expected_cumulative_state"], merged)
        targeted_results.append({
            "id":         case["id"],
            "category":   case["category"],
            "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1], "detail": ext_cmp},
            "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1], "detail": cum_cmp},
        })

    scenario_results = []
    for scenario in scenarios:
        state, history, turn_results = {f: None for f in CLINICAL_FIELDS}, [], []
        for turn in scenario["turns"]:
            msg       = turn["caregiver_message"]
            extracted = run_extraction_sc(msg, history, system_prompt)
            ext_cmp   = compare_fields(turn["expected_extracted_fields"], extracted)
            state     = simulate_merge(state, extracted)
            cum_cmp   = compare_fields(turn["expected_cumulative_state"], state)
            nf        = next_required_field(state)
            history.append(("caregiver", msg))
            if nf:
                history.append(("agent", QUESTION_MAP[nf]))
            turn_results.append({
                "turn":       turn["turn"],
                "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1]},
                "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1]},
            })
        scenario_results.append({"id": scenario["id"], "turns": turn_results})

    return {"targeted": targeted_results, "scenarios": scenario_results}


# ── Run: SC (llama-3.3-70b, temp=0.7, k=5, PROMPT_FULL) ────────────────────
print(f"Running SC (k={SC_K}) ...", end=" ", flush=True)
t0 = time.time()
result_sc  = evaluate_sc(PROMPT_FULL)
time_sc    = time.time() - t0
c = sum(r["extraction"]["correct"] for r in result_sc["targeted"])
t = sum(r["extraction"]["total"]   for r in result_sc["targeted"])
print(f"{c}/{t}  ({time_sc:.1f}s)")

Running SC (k=5) ... 120/130  (180.5s)


## Cell 6 — Baseline vs SC comparison table

In [20]:
# ── Side-by-side: Baseline vs SC ────────────────────────────────────────────
methods = [
    ("Baseline  (temp=0, k=1)",   result_baseline, time_baseline),
    (f"SC k={SC_K}   (temp=0.7)", result_sc,       time_sc),
]


def acc(result):
    c = sum(r["extraction"]["correct"] for r in result["targeted"])
    t = sum(r["extraction"]["total"]   for r in result["targeted"])
    return c, t


col     = 28
divider = "-" * (2 + col * len(methods))

print(f"\n{'METHOD COMPARISON — llama-3.3-70b + PROMPT_FULL':^{col * len(methods)}}")
print(divider)
print("".join(f"{lbl:^{col}}" for lbl, _, _ in methods))
print(divider)
print("".join(f"{f'{c}/{t} ({int(100*c/t)}%)':^{col}}"
              for lbl, res, _ in methods for c, t in [acc(res)]))
print("".join(f"{f'{tm:.1f}s':^{col}}" for _, _, tm in methods))
print(divider)

# Per-case breakdown
print(f"\n{'Case':<10} {'Category':<28}" + "".join(f"{lbl:^{col}}" for lbl, _, _ in methods))
print("-" * (38 + col * len(methods)))
for i, case in enumerate(targeted_cases):
    row = f"{case['id']:<10} {case['category']:<28}"
    for _, result, _ in methods:
        r = result["targeted"][i]
        c, t = r["extraction"]["correct"], r["extraction"]["total"]
        row += f"{str(c)+'/'+str(t)+(' ✓' if c==t else ' ✗'):^{col}}"
    print(row)

print("-" * (38 + col * len(methods)))
totals_row = f"{'TOTAL':<10} {'':28}"
time_row   = f"{'TIME':<10}  {'':27}"
for lbl, result, tm in methods:
    c, t = acc(result)
    totals_row += f"{str(c)+'/'+str(t)+' ('+str(int(100*c/t))+'%)':^{col}}"
    time_row   += f"{tm:.1f}s".center(col)
print(totals_row)
print(time_row)


    METHOD COMPARISON — llama-3.3-70b + PROMPT_FULL     
----------------------------------------------------------
  Baseline  (temp=0, k=1)       SC k=5   (temp=0.7)     
----------------------------------------------------------
       120/130 (92%)               120/130 (92%)        
            8.6s                       180.5s           
----------------------------------------------------------

Case       Category                      Baseline  (temp=0, k=1)       SC k=5   (temp=0.7)     
----------------------------------------------------------------------------------------------
tc_01      implied-field                         13/13 ✓                     13/13 ✓           
tc_02      ambiguous-value                       12/13 ✗                     12/13 ✗           
tc_03      non-overwrite-trap                    13/13 ✓                     13/13 ✓           
tc_04      medication-disambiguation             11/13 ✗                     11/13 ✗           
tc_05      hedge-l

## Cell 7 — Scenario results

In [21]:
print("SCENARIO RESULTS — Cumulative state accuracy per turn")
col = 28
scenario_methods = [
    ("Baseline  (temp=0, k=1)",   result_baseline),
    (f"SC k={SC_K}   (temp=0.7)", result_sc),
]
print(f"{'':30}" + "".join(f"{lbl:^{col}}" for lbl, _ in scenario_methods))
print("-" * (30 + col * len(scenario_methods)))

for s_idx, scenario in enumerate(scenarios):
    for t_idx, turn in enumerate(scenario["turns"]):
        label = f"{scenario['id']} turn {turn['turn']}"
        row = f"{label:<30}"
        for _, result in scenario_methods:
            tr = result["scenarios"][s_idx]["turns"][t_idx]
            c, t = tr["cumulative"]["correct"], tr["cumulative"]["total"]
            cell = f"{c}/{t}" + (" ✓" if c == t else " ✗")
            row += f"{cell:^{col}}"
        print(row)

SCENARIO RESULTS — Cumulative state accuracy per turn
                                Baseline  (temp=0, k=1)       SC k=5   (temp=0.7)     
--------------------------------------------------------------------------------------
scenario_1 turn 1                       12/13 ✗                     12/13 ✗           
scenario_1 turn 2                       11/13 ✗                     11/13 ✗           
scenario_1 turn 3                       11/13 ✗                     11/13 ✗           
scenario_2 turn 1                       12/13 ✗                     12/13 ✗           
scenario_2 turn 2                       13/13 ✓                     13/13 ✓           
scenario_2 turn 3                       13/13 ✓                     13/13 ✓           


## Cell 8 — Failure detail for any cell

In [22]:
# Inspect a specific method × case combination
INSPECT_METHOD = "baseline"   # "baseline" or "sc"
INSPECT_CASE   = "tc_01"      # tc_01 through tc_10

result      = result_baseline if INSPECT_METHOD == "baseline" else result_sc
case_result = next((r for r in result["targeted"] if r["id"] == INSPECT_CASE), None)

if case_result is None:
    print(f"Not found: {INSPECT_METHOD} / {INSPECT_CASE}")
else:
    label = "Baseline (temp=0, k=1)" if INSPECT_METHOD == "baseline" else f"SC k={SC_K} (temp=0.7)"
    print(f"Method: {label}  Case: {INSPECT_CASE}")
    print(f"Category: {case_result['category']}\n")
    print(f"{'Field':<25} {'Expected':<15} {'Actual':<15} Result")
    print("-" * 65)
    for field, r in case_result["extraction"]["detail"].items():
        mark = "OK  " if r["match"] else "FAIL"
        print(f"{field:<25} {str(r['expected']):<15} {str(r['actual']):<15} {mark}")

Method: Baseline (temp=0, k=1)  Case: tc_01
Category: implied-field

Field                     Expected        Actual          Result
-----------------------------------------------------------------
fever                     None            None            OK  
temperature_f             None            None            OK  
fever_duration_days       None            None            OK  
vomiting                  None            None            OK  
alert                     no              no              OK  
drinking                  None            None            OK  
urination_8h              None            None            OK  
breathing_issues          None            None            OK  
rash                      None            None            OK  
neck_pain                 None            None            OK  
age_months                None            None            OK  
current_medication        None            None            OK  
medication_last_dose      None            No